### Elastic properties



See this reference cite:PhysRevB.65.104104.

We seek the elastic constant tensor that relates stress (&sigma;) and strain (&epsilon;) via $\sigma = c \epsilon $. The stress and strain are six component vectors, so $c$ will be a 6 &times; 6 symmetric matrix.



#### Fe elastic properties



As with molecular vibrations, we need a groundstate geometry. Let us get one for BCC Fe.



In [1]:
from vasp import Vasp
from ase.lattice.cubic import BodyCenteredCubic

atoms = BodyCenteredCubic(symbol='Fe')
for atom in atoms:
    atom.magmom = 3.0

# from vasp.vasprc import VASPRC  # Deprecated in new vasp
# # VASPRC['mode'] = None  # Deprecated in new vasp  # Deprecated in new vasp

import logging
log = logging.getLogger('Vasp')
#log.setLevel(logging.DEBUG)

calc = Vasp(label='bulk/Fe-bulk',
            xc='PBE',
            kpts=[6, 6, 6],
            encut=350,
            ispin=2,
            isif=3,
            nsw=30,
            ibrion=1,
            atoms=atoms)
print(atoms.get_potential_energy())
print(atoms.get_stress())

-15.53472773
[ 0.00031141  0.00031141  0.00031141 -0.         -0.         -0.        ]

    -15.53472773
    [ 0.00031141  0.00031141  0.00031141 -0.         -0.         -0.        ]

Ok, now with a relaxed geometry at hand, we proceed with the elastic constants. This is accomplished with IBRION=6 and ISIF > 3 in VASP. See this reference (from the VASP [page](http://cms.mpi.univie.ac.at/vasp/vasp/IBRION_5_IBRION_6.html)) Y. Le Page and P. Saxe, Phys. Rev. B 65, 104104 (2002) [https://doi.org/10.1103/PhysRevB.65.104104>](https://doi.org/10.1103/PhysRevB.65.104104>)for more details.



In [1]:
from vasp import Vasp

calc = Vasp(label='bulk/Fe-bulk')
calc.clone('bulk/Fe-elastic')

calc.set(ibrion=6,    #
         isif=3,      # gets elastic constants
         potim=0.05,  # displacements
         nsw=1,
         nfree=2)

print(calc.potential_energy)

-15.52764065

    -15.52764065

Now, the results are written out to the OUTCAR file. Actually, three sets of moduli are written out 1) the elastic tensor for rigid ions, 2) the contribution from allowing the atoms to relax, and 3) the total elastic modulus, all in kBar.

     SYMMETRIZED ELASTIC MODULI (kBar)
    Direction    XX          YY          ZZ          XY          YZ          ZX
    --------------------------------------------------------------------------------
    XX        2803.5081   1622.6085   1622.6085      0.0000      0.0000      0.0000
    YY        1622.6085   2803.5081   1622.6085      0.0000      0.0000      0.0000
    ZZ        1622.6085   1622.6085   2803.5081      0.0000      0.0000      0.0000
    XY           0.0000      0.0000      0.0000    866.8792      0.0000      0.0000
    YZ           0.0000      0.0000      0.0000      0.0000    866.8792      0.0000
    ZX           0.0000      0.0000      0.0000      0.0000      0.0000    866.8792
    --------------------------------------------------------------------------------

and

    ELASTIC MODULI CONTR FROM IONIC RELAXATION (kBar)
    Direction    XX          YY          ZZ          XY          YZ          ZX
    --------------------------------------------------------------------------------
    XX           0.0000      0.0000      0.0000      0.0000      0.0000      0.0000
    YY           0.0000      0.0000      0.0000      0.0000      0.0000      0.0000
    ZZ           0.0000      0.0000      0.0000      0.0000      0.0000      0.0000
    XY           0.0000      0.0000      0.0000      0.0000      0.0000      0.0000
    YZ           0.0000      0.0000      0.0000      0.0000      0.0000      0.0000
    ZX           0.0000      0.0000      0.0000      0.0000      0.0000      0.0000
    --------------------------------------------------------------------------------
    
    
    TOTAL ELASTIC MODULI (kBar)
    Direction    XX          YY          ZZ          XY          YZ          ZX
    --------------------------------------------------------------------------------
    XX        2803.5081   1622.6085   1622.6085      0.0000      0.0000      0.0000
    YY        1622.6085   2803.5081   1622.6085      0.0000      0.0000      0.0000
    ZZ        1622.6085   1622.6085   2803.5081      0.0000      0.0000      0.0000
    XY           0.0000      0.0000      0.0000    866.8792      0.0000      0.0000
    YZ           0.0000      0.0000      0.0000      0.0000    866.8792      0.0000
    ZX           0.0000      0.0000      0.0000      0.0000      0.0000    866.8792
    --------------------------------------------------------------------------------

Let us write a small code here to extract the Total elastic moduli from the OUTCAR file. First we get the line where the total elastic moduli start, then take the six lines that start three lines after that. Finally we parse out the matrix elements and cast them as floats.



In [ ]:
import numpy as np
EM = []

with open('bulk/Fe-elastic/OUTCAR') as f:
    lines = f.readlines()
    for i, line in enumerate(lines):
        if line.startswith(' TOTAL ELASTIC MODULI (kBar)'):
            j = i + 3
            data = lines[j:j+6]
            break

for line in data:
    EM += [[float(x) for x in line.split()[1:]]]

print(np.array(EM))

    [[ 1125.1405  3546.8135  3546.8135     0.         0.         0.    ]
     [ 3546.8135  1125.1405  3546.8135     0.         0.         0.    ]
     [ 3546.8135  3546.8135  1125.1405     0.         0.         0.    ]
     [    0.         0.         0.      1740.2372     0.         0.    ]
     [    0.         0.         0.         0.      1740.2372     0.    ]
     [    0.         0.         0.         0.         0.      1740.2372]]

Fe is in a BCC crystal structure, which is high in symmetry. Consequently, many of the elements in the matrix are equal to zero.

See [http://www.nist.gov/data/PDFfiles/jpcrd34.pdf>](http://www.nist.gov/data/PDFfiles/jpcrd34.pdf>)for a lot of detail about Fe-Ni alloys and general theory about elastic constants. In the next section, we show how the code above is integrated into mod:Vasp.



#### Al elastic properties



First, the relaxed geometry.



In [1]:
from vasp import Vasp
from ase.lattice.cubic import FaceCenteredCubic

atoms = FaceCenteredCubic(symbol='Al')

calc = Vasp(label='bulk/Al-bulk',
            xc='PBE',
            kpts=[12, 12, 12],
            encut=350,
            prec='High',
            isif=3,
            nsw=30,
            ibrion=1,
            atoms=atoms)
print(calc.potential_energy)

-14.97511793

    -14.97511793

Ok, now with a relaxed geometry at hand, we proceed with the elastic constants. This is accomplished with IBRION=6 and ISIF &ge; 3 in VASP.



In [1]:
from vasp import Vasp

calc = Vasp(label='bulk/Al-bulk')
calc.clone('bulk/Al-elastic')

calc.set(ibrion=6,    #
         isif=3,      # gets elastic constants
         potim=0.015,  # displacements
         nsw=1,
         nfree=2)

calc.wait(abort=True)

EM = calc.get_elastic_moduli()

print(EM)

c11 = EM[0, 0]
c12 = EM[0, 1]
B = (c11 + 2 * c12) / 3.0
print(B)

[[ 110.17099   59.54652   59.54652    0.         0.         0.     ]
 [  59.54652  110.17099   59.54652    0.         0.         0.     ]
 [  59.54652   59.54652  110.17099    0.         0.         0.     ]
 [   0.         0.         0.        11.52331    0.         0.     ]
 [   0.         0.         0.         0.        11.52331    0.     ]
 [   0.         0.         0.         0.         0.        11.52331]]
76.4213433333

    [[ 110.17099   59.54652   59.54652    0.         0.         0.     ]
     [  59.54652  110.17099   59.54652    0.         0.         0.     ]
     [  59.54652   59.54652  110.17099    0.         0.         0.     ]
     [   0.         0.         0.        11.52331    0.         0.     ]
     [   0.         0.         0.         0.        11.52331    0.     ]
     [   0.         0.         0.         0.         0.        11.52331]]
    76.4213433333

This example shows the basic mechanics of getting the elastic constants. The C<sub>44</sub> constant above is too low, and probably we need to check these constants for convergence with respect to kpoints, planewave cutoff, and maybe the value of POTIM.



#### Manual calculation of elastic constants



It is possible to manually calculate single elastic constants; you just need to know what strain corresponds to the elastic constant.

For the C11 elastic constant in a cubic system, we simply strain the cell along the x-axis, and then evaluate the second derivative at the minimum to calculate C11 like this.

$C_{11} = \frac{1}{V_C}\frac{\partial^2 E^{tot}}{\partial \gamma^2}$



In [1]:
from vasp import Vasp
from ase.lattice.cubic import FaceCenteredCubic
import numpy as np
import matplotlib.pyplot as plt

DELTAS = np.linspace(-0.05, 0.05, 5)
calcs = []
volumes = []

for delta in DELTAS:
    atoms = FaceCenteredCubic(symbol='Al')

    cell = atoms.cell

    T = np.array([[1 + delta, 0, 0],
                  [0,1, 0],
                  [0, 0, 1]])
    newcell = np.dot(cell, T)
    atoms.set_cell(newcell, scale_atoms=True)
    volumes += [atoms.get_volume()]

    calcs += [Vasp(label='bulk/Al-c11-{}'.format(delta),
                xc='pbe',
                kpts=[12, 12, 12],
                encut=350,
                atoms=atoms)]

Vasp.run()
energies =  [calc.potential_energy for calc in calcs]

# fit a parabola
eos = np.polyfit(DELTAS, energies, 2)

# first derivative
d_eos = np.polyder(eos)

print(np.roots(d_eos))

xfit = np.linspace(min(DELTAS), max(DELTAS))
yfit = np.polyval(eos, xfit)

plt.plot(DELTAS, energies, 'bo', xfit, yfit, 'b-')
plt.xlabel('$\delta$')
plt.ylabel('Energy (eV)')
plt.savefig('images/Al-c11.png')

[ 0.00727102]

![img](./images/Al-c11.png)

